In [4]:
from model import TinyHandTracker

net = TinyHandTracker()

In [ ]:
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset


class YoloLabelDataset(Dataset):
    def __init__(self, images_dir: Path, labels_dir: Path, transform=None):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.transform = transform

        allowed_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        self.samples = []

        for image_path in sorted(self.images_dir.iterdir()):
            if image_path.suffix.lower() not in allowed_ext:
                continue

            label_path = self.labels_dir / f"{image_path.stem}.txt"
            if label_path.exists():
                self.samples.append((image_path, label_path))

        if not self.samples:
            raise RuntimeError(f"Nu exista perechi imagine-label in {self.images_dir}")

    def __len__(self):
        return len(self.samples)

    def _read_label(self, label_path: Path):
        with open(label_path, "r") as f:
            first_line = next((line.strip() for line in f if line.strip()), None)

        if first_line is None:
            raise ValueError(f"Label gol: {label_path}")

        parts = first_line.replace(",", " ").split()
        
        # Formatul tău: [clasa, x, y, w, h]
        class_id = float(parts[0]) # 0 = mână, 1 = nu e mână
        x = float(parts[1])
        y = float(parts[2])
        w = float(parts[3])
        h = float(parts[4])
        
        # Transformăm clasa într-o probabilitate: 1.0 (mână) sau 0.0 (fundal)
        probabilitate_mana = 1.0 if class_id == 0.0 else 0.0
        
        # Returnăm formatul standardizat pe care îl va învăța rețeaua
        return torch.tensor([x, y, w, h, probabilitate_mana], dtype=torch.float32)

    def __getitem__(self, idx):
        image_path, label_path = self.samples[idx]

        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        label = self._read_label(label_path)
        return image, label



# Config
dataset_root = Path("dataset/filteredDataset")
batch_size = 32
image_size = 128

# Transformari
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

# Foldere train/test
train_images_dir = dataset_root / "train" / "images_filtered"
train_labels_dir = dataset_root / "train" / "labels" / "YOLO_filtered"
test_images_dir = dataset_root / "test" / "images_filtered"
test_labels_dir = dataset_root / "test" / "labels" / "YOLO_filtered"

# Dataset-uri
train_dataset = YoloLabelDataset(train_images_dir, train_labels_dir, transform=transform)
test_dataset = YoloLabelDataset(test_images_dir, test_labels_dir, transform=transform)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Info util
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")


Train samples: 862
Test samples: 221


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Antrenamentul rulează pe: {device}")

# -----------------------------------------------------------------------------
# 1. Funcția de Pierdere Personalizată (Loss Function)
# -----------------------------------------------------------------------------
import torch.nn as nn
import torch.nn.functional as F

class HandTrackerLoss(nn.Module):
    def __init__(self):
        super(HandTrackerLoss, self).__init__()

    def forward(self, predictii, etichete):
        # Formatul este fix: [x, y, w, h, conf]
        
        # Decupăm predicțiile modelului
        pred_box = predictii[:, :4]  # Ia indicii 0, 1, 2, 3
        pred_conf = predictii[:, 4]  # Ia doar ultimul index

        # Decupăm etichetele corecte
        target_box = etichete[:, :4]
        target_conf = etichete[:, 4] # Deja formatat ca 1.0 sau 0.0

        # 1. Eroarea de clasificare (învață să recunoască mâna vs fundal)
        loss_conf = F.binary_cross_entropy_with_logits(pred_conf, target_conf)

        # 2. Eroarea pe coordonate (ignorată dacă target_conf este 0)
        loss_box_brut = F.mse_loss(pred_box, target_box, reduction='none').mean(dim=1) 
        loss_box_mascat = (loss_box_brut * target_conf).mean()

        # Eroarea totală
        return loss_conf + (loss_box_mascat * 5.0)

# -----------------------------------------------------------------------------
# 2. Funcția de Evaluare (Înlocuiește test_acc)
# -----------------------------------------------------------------------------

def evaluate_model(net: nn.Module, test_loader: DataLoader, loss_fn: nn.Module):
    net.eval() # Trecem rețeaua în modul de testare (dezactivează anumite straturi ca Dropout, dacă există)
    total_loss = 0.0
    batches = 0

    # Oprim calculul gradienților pentru a salva memorie și timp
    with torch.no_grad():
        for test_images, test_labels in test_loader:
            test_images = test_images.to(device) # Mutăm pe GPU
            test_labels = test_labels.to(device)
            out = net(test_images)
            loss = loss_fn(out, test_labels)
            total_loss += loss.item()
            batches += 1

    # Returnăm eroarea medie pe setul de testare
    return total_loss / batches if batches > 0 else 0

# -----------------------------------------------------------------------------
# 3. Bucla Principală de Antrenare
# -----------------------------------------------------------------------------
def train_fn(epochs: int, train_loader: DataLoader, test_loader: DataLoader,
             net: nn.Module, loss_fn: nn.Module, optimizer: optim.Optimizer):
    bestValLoss = float('inf')

    for e in range(epochs):
        net.train() # Trecem rețeaua înapoi în modul de antrenare
        train_loss_acumulat = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # 1. Resetăm gradienții (mutat la început!)
            optimizer.zero_grad() 
            
            # 2. Trecem datele prin rețea
            out = net(images)
            
            # 3. Calculăm cât de mult a greșit
            loss = loss_fn(out, labels)
            
            # 4. Calculăm corecțiile (Backpropagation)
            loss.backward()
            
            # 5. Aplicăm corecțiile filtrelor
            optimizer.step()

            train_loss_acumulat += loss.item()

        # Calculăm mediile pentru afișare
        avg_train_loss = train_loss_acumulat / len(train_loader)
        val_loss = evaluate_model(net, test_loader, loss_fn)

        if val_loss < bestValLoss:
            bestValLoss = val_loss
            torch.save(net.state_dict(), 'best_model.pth')

        if(val_loss - avg_train_loss > 0.15):
            print(f"Overfittiing, ma opresc")
            return

        print(f"Epoca {e + 1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Validation Loss: {val_loss:.4f}")

# -----------------------------------------------------------------------------
# 4. Execuția Antrenamentului
# -----------------------------------------------------------------------------

# Presupunem că ai definit deja `TinyHandTracker` și dataloader-ele
net = TinyHandTracker().to(device)

# Folosim Adam, un optimizator excelent pentru astfel de rețele
optimizer = optim.Adam(net.parameters(), lr=0.001)

# Folosim funcția noastră de pierdere
loss_fn = HandTrackerLoss()

# Pornim antrenamentul pentru 20 de epoci
print("Incepem antrenamentul...")
train_fn(epochs=100, 
            train_loader=train_loader, 
            test_loader=test_loader, 
            net=net, 
            loss_fn=loss_fn, 
            optimizer=optimizer)

Antrenamentul rulează pe: cpu
Incepem antrenamentul...
Epoca 1/100 | Train Loss: 0.8643 | Validation Loss: 0.6911
Epoca 2/100 | Train Loss: 0.6282 | Validation Loss: 0.5993
Epoca 3/100 | Train Loss: 0.6063 | Validation Loss: 0.6079
Epoca 4/100 | Train Loss: 0.6076 | Validation Loss: 0.6102
Epoca 5/100 | Train Loss: 0.5980 | Validation Loss: 0.5812
Epoca 6/100 | Train Loss: 0.5939 | Validation Loss: 0.5961
Epoca 7/100 | Train Loss: 0.6046 | Validation Loss: 0.5819
Epoca 8/100 | Train Loss: 0.5962 | Validation Loss: 0.5807
Epoca 9/100 | Train Loss: 0.5907 | Validation Loss: 0.5755
Epoca 10/100 | Train Loss: 0.5848 | Validation Loss: 0.5712
Epoca 11/100 | Train Loss: 0.5860 | Validation Loss: 0.5638
Epoca 12/100 | Train Loss: 0.5728 | Validation Loss: 0.5958
Epoca 13/100 | Train Loss: 0.5831 | Validation Loss: 0.5455
Epoca 14/100 | Train Loss: 0.5890 | Validation Loss: 0.5378
Epoca 15/100 | Train Loss: 0.5625 | Validation Loss: 0.5630
Epoca 16/100 | Train Loss: 0.5671 | Validation Loss: 0

KeyboardInterrupt: 

In [18]:
model = TinyHandTracker()
model.load_state_dict(torch.load("best_model.pth", weights_only=True))


print(1 - evaluate_model(model, test_loader, loss_fn))

0.5516065614564079
